In [ ]:
import pandas as pd
import numpy as np

# Configuração
np.random.seed(42)
datas = pd.date_range(start="2024-01-01", end="2024-12-31", freq="D")
total_clientes = 100_000

# Funções para representar condições externas e sazonalidade do varejo
def clima_loja_fisica(data):
    mes = data.month
    temperatura = 24 + 6 * np.sin((mes - 1) / 12 * 2 * np.pi) + np.random.normal(0, 1.5)
    chuva = np.random.binomial(1, 0.35 if mes in [11, 12, 1, 2, 3] else 0.15)
    return temperatura, chuva


def clima_ecommerce(data):
    mes = data.month
    temperatura = 22 + 4 * np.sin((mes - 1) / 12 * 2 * np.pi) + np.random.normal(0, 1.5)
    chuva = np.random.binomial(1, 0.30 if mes in [11, 12, 1, 2, 3] else 0.20)
    return temperatura, chuva


def clientes_loja_fisica(data):
    mes = data.month
    base = 180 + 80 * np.sin((mes - 1) / 12 * 2 * np.pi)
    if data.weekday() >= 5:
        base *= 1.2
    if mes in [1, 12]:
        base *= 1.5
    return max(50, int(np.random.normal(base, 20)))


def clientes_ecommerce(data):
    mes = data.month
    base = 150 + 65 * np.sin((mes - 1) / 12 * 2 * np.pi)
    if data.weekday() >= 5:
        base *= 1.3
    if mes in [11, 12]:
        base *= 1.4
    return max(40, int(np.random.normal(base, 18)))


# Criar DataFrame diário para os canais de venda
registros = []

for data in datas:
    promocao = int(data.weekday() in [4, 5] or data.month in [11, 12])

    temperatura, chuva = clima_loja_fisica(data)
    clientes = clientes_loja_fisica(data)
    categoria_a = int(clientes * np.clip((temperatura - 20) / 15, 0, 1) * (1 - 0.3 * chuva))
    categoria_b = int(clientes * np.clip((temperatura - 22) / 13, 0, 1) * (1 - 0.4 * chuva))
    pedidos = int(clientes * (0.70 + 0.10 * promocao))
    demanda_produto = max(0, int(pedidos * (0.30 + 0.02 * temperatura + 0.10 * promocao - 0.10 * chuva) + np.random.normal(0, 2)))
    registros.append([data, "loja_fisica", clientes, categoria_a, categoria_b, pedidos, temperatura, chuva, promocao, demanda_produto])

    temperatura, chuva = clima_ecommerce(data)
    clientes = clientes_ecommerce(data)
    categoria_a = int(clientes * (0.45 + 0.15 * promocao) * (1 - 0.10 * chuva))
    categoria_b = int(clientes * (0.35 + 0.10 * promocao) * (1 - 0.05 * chuva))
    pedidos = int(clientes * (0.80 + 0.12 * promocao))
    demanda_produto = max(0, int(pedidos * (0.35 + 0.015 * (25 - temperatura) + 0.12 * promocao) + np.random.normal(0, 2)))
    registros.append([data, "ecommerce", clientes, categoria_a, categoria_b, pedidos, temperatura, chuva, promocao, demanda_produto])

df = pd.DataFrame(registros, columns=[
    "data", "canal_venda", "clientes", "categoria_a", "categoria_b",
    "pedidos", "temperatura_media", "chuva", "promocao", "demanda_produto"
])

# Ajustar a escala para aproximadamente 100 mil clientes no ano
fator = total_clientes / df["clientes"].sum()
df["clientes"] = (df["clientes"] * fator).astype(int)

# Salvar os dados sintéticos do projeto
csv_path = "dados_sinteticos_varejo.csv"
df.to_csv(csv_path, index=False)

csv_path

In [ ]:
df = pd.read_csv("dados_sinteticos_varejo.csv", parse_dates=["data"])
df

# Análise de correlação

Gere um heatmap da matriz de correlação das variáveis numéricas do dataframe `df` para identificar relações entre clientes, pedidos, promoções e demanda de produto.

## Calcular a matriz de correlação

A matriz será calculada somente para as colunas numéricas do dataframe.

## Visualizar o heatmap

O heatmap facilita a identificação de relações positivas e negativas entre as variáveis de vendas e demanda.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number)
correlation_matrix = numeric_cols.corr()
display(correlation_matrix)

## Visualizar o heatmap

### Subtask:
Gere um heatmap da matriz de correlação usando uma biblioteca de visualização.


**Reasoning**:
Generate a heatmap of the correlation matrix using seaborn and matplotlib.



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Heatmap da matriz de correlação do varejo")
plt.tight_layout()
plt.show()

## Preparação dos dados para modelagem

Os dados serão separados por canal de venda. Variáveis sazonais serão adicionadas para capturar padrões ao longo do ano.

In [ ]:
from sklearn.model_selection import train_test_split

# Separar os canais de venda e criar variáveis sazonais
df_loja_fisica = df[df["canal_venda"] == "loja_fisica"].copy()
df_ecommerce = df[df["canal_venda"] == "ecommerce"].copy()

for dados_canal in [df_loja_fisica, df_ecommerce]:
    dia_ano = dados_canal["data"].dt.dayofyear
    dados_canal["seno_dia_ano"] = np.sin(2 * np.pi * dia_ano / 365)
    dados_canal["cosseno_dia_ano"] = np.cos(2 * np.pi * dia_ano / 365)

colunas_excluidas = ["data", "canal_venda", "demanda_produto"]
X_loja_fisica = df_loja_fisica.drop(columns=colunas_excluidas)
y_loja_fisica = df_loja_fisica["demanda_produto"]
X_ecommerce = df_ecommerce.drop(columns=colunas_excluidas)
y_ecommerce = df_ecommerce["demanda_produto"]

X_train_loja_fisica, X_test_loja_fisica, y_train_loja_fisica, y_test_loja_fisica = train_test_split(
    X_loja_fisica, y_loja_fisica, test_size=0.2, random_state=0
)
X_train_ecommerce, X_test_ecommerce, y_train_ecommerce, y_test_ecommerce = train_test_split(
    X_ecommerce, y_ecommerce, test_size=0.2, random_state=0
)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

modelos_loja_fisica = {
    "Regressão Linear": LinearRegression(),
    "Árvore de Decisão": DecisionTreeRegressor(random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
}
modelos_ecommerce = {
    "Regressão Linear": LinearRegression(),
    "Árvore de Decisão": DecisionTreeRegressor(random_state=42),
    "XGBoost": XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
}

for modelo in modelos_loja_fisica.values():
    modelo.fit(X_train_loja_fisica, y_train_loja_fisica)
for modelo in modelos_ecommerce.values():
    modelo.fit(X_train_ecommerce, y_train_ecommerce)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

resultados = []
for canal, modelos, X_test, y_test in [
    ("loja_fisica", modelos_loja_fisica, X_test_loja_fisica, y_test_loja_fisica),
    ("ecommerce", modelos_ecommerce, X_test_ecommerce, y_test_ecommerce),
]:
    for nome_modelo, modelo in modelos.items():
        previsoes = modelo.predict(X_test)
        resultados.append({
            "canal_venda": canal,
            "modelo": nome_modelo,
            "RMSE": mean_squared_error(y_test, previsoes) ** 0.5,
            "MAE": mean_absolute_error(y_test, previsoes),
            "R2": r2_score(y_test, previsoes),
        })

resultados_df = pd.DataFrame(resultados)
resultados_df

In [ ]:
# Os modelos dos dois canais foram treinados na célula anterior.

In [ ]:
# As métricas dos modelos dos dois canais foram calculadas na célula anterior.

In [ ]:
print("Métricas de desempenho por canal de venda:")
display(resultados_df.sort_values(["canal_venda", "RMSE"]))

## Validação cruzada

A validação cruzada compara a estabilidade dos três modelos nos canais de loja física e e-commerce.

In [ ]:
# A validação cruzada consolidada será calculada no próximo bloco.

### Validação Cruzada para outros Modelos (Praia e Fazenda)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
validacao_cruzada = []

for canal, modelos, X_treino, y_treino in [
    ("loja_fisica", modelos_loja_fisica, X_train_loja_fisica, y_train_loja_fisica),
    ("ecommerce", modelos_ecommerce, X_train_ecommerce, y_train_ecommerce),
]:
    for nome_modelo, modelo in modelos.items():
        scores = cross_val_score(modelo, X_treino, y_treino, cv=kf, scoring="neg_mean_squared_error")
        validacao_cruzada.append({
            "canal_venda": canal,
            "modelo": nome_modelo,
            "RMSE_medio": np.sqrt(-scores).mean(),
        })

validacao_df = pd.DataFrame(validacao_cruzada)
display(validacao_df.sort_values(["canal_venda", "RMSE_medio"]))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df_loja_fisica, x="clientes", y="demanda_produto", alpha=0.6, ax=axes[0])
axes[0].set_title("Loja física: clientes x demanda")
axes[0].set_xlabel("Clientes")
axes[0].set_ylabel("Demanda do produto")

sns.scatterplot(data=df_ecommerce, x="clientes", y="demanda_produto", alpha=0.6, ax=axes[1])
axes[1].set_title("E-commerce: clientes x demanda")
axes[1].set_xlabel("Clientes")
axes[1].set_ylabel("Demanda do produto")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

TEST_DAYS = 90
COST_PER_UNIT = 5.0
SHORTAGE_PENALTY = 12.0
SAFETY_MARGIN = 0.05

feature_cols = [
    "clientes", "categoria_a", "categoria_b", "pedidos",
    "temperatura_media", "chuva", "promocao", "sin_dia_ano",
    "cos_dia_ano", "is_weekend", "month"
]

dados_modelagem = df.copy()
dados_modelagem["dia_ano"] = dados_modelagem["data"].dt.dayofyear
dados_modelagem["sin_dia_ano"] = np.sin(2 * np.pi * dados_modelagem["dia_ano"] / 365)
dados_modelagem["cos_dia_ano"] = np.cos(2 * np.pi * dados_modelagem["dia_ano"] / 365)
dados_modelagem["weekday"] = dados_modelagem["data"].dt.weekday
dados_modelagem["is_weekend"] = (dados_modelagem["weekday"] >= 5).astype(int)
dados_modelagem["month"] = dados_modelagem["data"].dt.month


def simular_custos(comprado, real):
    comprado = np.asarray(comprado)
    real = np.asarray(real)
    excesso = np.maximum(0, comprado - real)
    falta = np.maximum(0, real - comprado)
    return {
        "excesso": excesso,
        "falta": falta,
        "custo_total": comprado * COST_PER_UNIT + falta * SHORTAGE_PENALTY,
    }


resumo_estoque = []
for canal in dados_modelagem["canal_venda"].unique():
    dados_canal = dados_modelagem[dados_modelagem["canal_venda"] == canal].reset_index(drop=True)
    treino = dados_canal.iloc[:-TEST_DAYS]
    teste = dados_canal.iloc[-TEST_DAYS:]

    modelo = XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
    modelo.fit(treino[feature_cols], treino["demanda_produto"])
    previsoes = np.maximum(modelo.predict(teste[feature_cols]), 0)
    media_dia_semana = treino.groupby("weekday")["demanda_produto"].mean()
    baseline = teste["weekday"].map(media_dia_semana).fillna(treino["demanda_produto"].mean()).to_numpy()
    demanda_real = teste["demanda_produto"].to_numpy()

    simulacao_modelo = simular_custos(np.ceil(previsoes * (1 + SAFETY_MARGIN)), demanda_real)
    simulacao_baseline = simular_custos(np.ceil(baseline * (1 + SAFETY_MARGIN)), demanda_real)
    simulacao_otima = simular_custos(demanda_real, demanda_real)

    resumo_estoque.append({
        "canal_venda": canal,
        "custo_otimo": simulacao_otima["custo_total"].sum(),
        "custo_modelo": simulacao_modelo["custo_total"].sum(),
        "custo_baseline": simulacao_baseline["custo_total"].sum(),
        "excesso_modelo_unidades": simulacao_modelo["excesso"].sum(),
        "falta_modelo_unidades": simulacao_modelo["falta"].sum(),
        "mae_modelo": mean_absolute_error(demanda_real, previsoes),
    })

resumo_estoque_df = pd.DataFrame(resumo_estoque)
resumo_estoque_df["diferenca_modelo_vs_otimo"] = resumo_estoque_df["custo_modelo"] - resumo_estoque_df["custo_otimo"]
display(resumo_estoque_df)

In [ ]:
# Os gráficos exploratórios por canal são apresentados nas células seguintes.

In [ ]:
import os

OUT_DIR = "exportacoes_varejo"
os.makedirs(OUT_DIR, exist_ok=True)

# Exportar medias moveis por canal para uso em dashboards
for canal in df["canal_venda"].unique():
    dados_canal = df[df["canal_venda"] == canal].sort_values("data").copy()
    dados_canal["clientes_media_movel"] = dados_canal["clientes"].rolling(7, min_periods=1).mean().round(2)
    dados_canal["demanda_media_movel"] = dados_canal["demanda_produto"].rolling(7, min_periods=1).mean().round(2)
    caminho = os.path.join(OUT_DIR, f"medias_moveis_{canal}.csv")
    dados_canal[["data", "canal_venda", "clientes", "clientes_media_movel", "demanda_produto", "demanda_media_movel"]].to_csv(caminho, index=False)
    print(f"Arquivo gerado: {caminho}")

resumo_estoque_df.to_csv(os.path.join(OUT_DIR, "resumo_estoque.csv"), index=False)
print(f"Arquivo gerado: {os.path.join(OUT_DIR, 'resumo_estoque.csv')}")

In [ ]:
import matplotlib.pyplot as plt

serie_vendas = df.sort_values("data").copy()
serie_vendas["demanda_media_movel"] = serie_vendas.groupby("canal_venda")["demanda_produto"].transform(
    lambda valores: valores.rolling(7, min_periods=1).mean()
)

plt.figure(figsize=(12, 6))
for canal, dados_canal in serie_vendas.groupby("canal_venda"):
    plt.plot(dados_canal["data"], dados_canal["demanda_media_movel"], label=canal.replace("_", " ").title())

plt.title("Demanda de produto por canal de venda")
plt.xlabel("Data")
plt.ylabel("Demanda média móvel de 7 dias")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Tabela final para apoiar a leitura dos resultados
resultado_final = resultados_df.merge(
    validacao_df,
    on=["canal_venda", "modelo"],
    how="left"
)
display(resultado_final.sort_values(["canal_venda", "RMSE"]))

In [ ]:
# Bloco reservado para futuras análises específicas por categoria de produto.